In [ ]:
import * as tslab from "tslab";
import { readFileSync } from "fs";

const css = readFileSync("../style.css", "utf-8");
tslab.display.html(`<style>${css}</style>`);

# Ordered Binary Trees

This notebook implements *ordered binary trees*.  In order to define this notion, we first have to define 
the concept of *ordered binary trees*.  In the following, assume a set $\texttt{Key}$ and a set $\texttt{Value}$ are given.   Then, the
set $\mathcal{B}$ of all *ordered binary trees* is defined inductively.
  - $\texttt{Nil} \in \mathcal{B}$
  - $\texttt{Node}(k, v, l, r) \in \mathcal{B}$  iff the following conditions hold:
      - $k \in\texttt{Key}$,
      - $v \in\texttt{Value}$,
      - $l \in\mathcal{B}$,
      - $r \in\mathcal{B}$,
      - all keys that occur in the left subtree $l$ are smaller than $k$,
      - all keys that occur in the right subtree $r$ are bigger than $k$,
      
        therefore $l < k < r$.

The class `OrderedBinaryTree` represents the nodes of an ordered binary tree.
  - $\texttt{Nil}$           is created as $\texttt{OrderedBinaryTree}()$.
  - $\texttt{Node}(k,v,l,r)$ is created as follows:
    ```
    t = OrderedBinaryTree()
    t.mKey   = k
    t.mValue = v
    t.mLeft  = l
    t.mRight = r
    ```
The constructor below creates the empty tree.

In [ ]:
class OrderedBinaryTree<Key, Value> {
    mKey: Key | null;
    mValue: Value | null;
    mLeft: OrderedBinaryTree<Key, Value> | null;
    mRight: OrderedBinaryTree<Key, Value> | null;
    mID: number;
    static sNodeCount: number = 0;
    constructor() {
        this.mKey = null;
        this.mValue = null;
        this.mLeft = null;
        this.mRight = null;
    }
}
interface OrderedBinaryTree<Key, Value> {
    mKey: Key | null;
    mValue: Value | null;
    mLeft: OrderedBinaryTree<Key, Value> | null;
    mRight: OrderedBinaryTree<Key, Value> | null;
    isEmpty(): boolean;
    insert(key: Key, value: Value): void;
    find(key: Key): Value | undefined;
    delete(key: Key): void;
    toString(): string;
    toDot(): string;
    _delMin(): [OrderedBinaryTree<Key, Value>, Key, Value];
    _update(tree: OrderedBinaryTree<Key, Value>): void;
    _keyArray(): Key[];
    _assignIDs(nodeDict: Record<number, OrderedBinaryTree<Key, Value>>): void;
}

Given an ordered binary tree $t$, the expression $t.\texttt{isEmpty}()$ checks whether $t$ is the empty tree.

In [ ]:
OrderedBinaryTree.prototype.isEmpty = function<Key, Value>(this: OrderedBinaryTree<Key, Value>): boolean {
    return this.mKey === null;
};

Given an ordered binary tree $t$ and a key $k$, the expression $t.\texttt{find}(k)$ returns the value stored unter the key $k$.
The method `find` is defined inductively as follows:
  - $\texttt{Nil}.\texttt{find}(k) = \Omega$,

    because the empty tree is interpreted as the empty map.
  - $\texttt{Node}(k, v, l, r).\texttt{find}(k) = v$,
      
    because the node $\texttt{Node}(k,v,l,r)$ stores the assignment $k \mapsto v$.
  - $k_1 < k_2 \rightarrow \texttt{Node}(k_2, v, l, r).\texttt{find}(k_1) = l.\texttt{find}(k_1)$,

    because if $k_1$ is less than $k_2$, then any mapping for $k_1$ has to be stored in the left subtree  $l$.
  - $k_1 > k_2 \rightarrow \texttt{Node}(k_2, v, l, r).\texttt{find}(k_1) = r.\texttt{find}(k_1)$,

    because if $k_1$ is greater than $k_2$, then any mapping for $k_1$ has to be stored in the right subtree  $r$.

In [ ]:
OrderedBinaryTree.prototype.find = function<Key, Value>(this: OrderedBinaryTree<Key, Value>, key: Key): Value | undefined {
    if (this.isEmpty()) {
        return undefined;
    } else if (this.mKey === key) {
        return this.mValue;
    } else if (key < this.mKey) {
        return this.mLeft.find(key);
    } else {
        return this.mRight.find(key);
    }
};

Given an ordered binary tree $t$, a key $k$ and a value $v$, the expression $t.\texttt{insert}(k, v)$ updates the tree $t$ such that the key $k$ is associated with the value $v$.
The method `insert` is defined inductively as follows:
  - $\texttt{Nil}.\texttt{insert}(k,v) = \texttt{Node}(k,v, \texttt{Nil}, \texttt{Nil})$,
  
    If the tree is empty, the information to be stored is stored at the root.
  - $\texttt{Node}(k,v_2,l,r).\texttt{insert}(k,v_1) = \texttt{Node}(k, v_1, l, r)$,

    If the key $k$ is located at the root, we overwrite the old information. 
  - $k_1 < k_2 \rightarrow 
    \texttt{Node}(k_2, v_2, l, r).\texttt{insert}(k_1, v_1) = \texttt{Node}\bigl(k_2, v_2, l.\texttt{insert}(k_1, v_1), r\bigr)$,

    If the key $k_1$, which is the key for which we want to store a value, is less than the key
    $k_2$ at the root, then we have to insert the information in the left subtree.
  - $k_1 > k_2 \rightarrow 
         \texttt{Node}(k_2, v_2, l, r).\texttt{insert}(k_1, v_1) = 
         \texttt{Node}\bigl(k_2, v_2, l, r.\texttt{insert}(k_1, v_1)\bigr)$,

    If the key $k_1$, which is the key for which we want to store a value, is bigger than the key
    $k_2$ at the root, then we have to insert the information in the right subtree.

In [ ]:
OrderedBinaryTree.prototype.insert = function<Key, Value>(this: OrderedBinaryTree<Key, Value>, key: Key, value: Value): void {
    if (this.isEmpty()) {
        this.mKey = key;
        this.mValue = value;
        this.mLeft = new OrderedBinaryTree<Key, Value>();
        this.mRight = new OrderedBinaryTree<Key, Value>();
    } else if (this.mKey === key) {
        this.mValue = value;
    } else if (key < this.mKey) {
        this.mLeft.insert(key, value);
    } else {
        this.mRight.insert(key, value);
    }
};

Given an ordered binary tree $t$ and a key $k$, the expression $t.\texttt{delete}(k)$ removes the key $k$ and its associated value from $t$.  The method `delete` is defined inductively.
  - $\texttt{Nil}.\texttt{delete}(k) = \texttt{Nil}$.
  - $\texttt{Node}(k,v,\texttt{Nil},r).\texttt{delete}\bigl(k\bigr) = r$.
  - $\texttt{Node}(k,v,l,\texttt{Nil}).\texttt{delete}(k) = l$.
  - If $l \not= \texttt{Nil} \,\wedge\, r \not= \texttt{Nil} \,\wedge\, r.\texttt{delMin}() = [r',k_{min}, v_{min}]$,
    then
    
    $$\texttt{Node}(k,v,l,r).\texttt{delete}(k) = \texttt{Node}(k_{min},v_{min},l,r').$$
      
    If the key to be removed is found at the root of the tree and neither of its subtrees is
    empty, the call  $r\mathtt{.}\texttt{delMin}()$ removes the smallest key together with its
    associated value from the subtree $r$ yielding the subtree $r'$.
    The smallest key from $r$ is then stored at the root of the new tree.

  - $k_1 < k_2 \rightarrow \texttt{Node}(k_2,v_2,l,r).\texttt{delete}\bigl(k_1) = 
    \texttt{Node}(k_2,v_2,l.\texttt{delete}(k_1),r)$.

    If the key that is to be removed is less than the key stored at the root, the key $k$ can only be
    located in the left subtree $l$.  Hence, $k$ is removed from the left subtree $l$ recursively.
  - $k_1 > k_2 \rightarrow \texttt{Node}(k_2,v_2,l,r).\texttt{delete}(k_1) = 
    \texttt{Node}(k_2,v_2,l,r.\texttt{delete}(k_1))$.

    If the key that is to be removed is greater than the key stored at the root, the key $k$ can only be
    located in the right subtree $r$.  Hence, $k$ is removed from the right subtree $r$ recursively.

In [ ]:
OrderedBinaryTree.prototype.delete = function<Key, Value>(this: OrderedBinaryTree<Key, Value>, key: Key): void {
    if (this.isEmpty()) {
        return;
    }
    if (key === this.mKey) {
        if (this.mLeft.isEmpty()) {
            this._update(this.mRight);
        } else if (this.mRight.isEmpty()) {
            this._update(this.mLeft);
        } else {
            const [rs, km, vm] = this.mRight._delMin();
            this.mKey = km;
            this.mValue = vm;
            this.mRight = rs;
        }
    } else if (key < this.mKey) {
        this.mLeft.delete(key);
    } else {
        this.mRight.delete(key);
    }
};

Given a non-empty ordered binary tree $t$, the expression $t.\texttt{delMin}()$ removes the smallest key $k_m$ and its associated value $v_m$ from $t$ and returns the triple
$$(r,k_m,v_m),$$
where $r$ is the tree that  results from removing $k_m$ and $v_m$ from $t$.  The function is defined via the following equations:
  - $\texttt{Node}(k, v, \texttt{Nil}, r).\texttt{delMin}() = (r, k, v)$

    If the left subtree is empty, $k$ has to be the smallest key in the tree 
    $\texttt{Node}(k, v, \texttt{Nil}, r)$.  If $k$ is removed, we are left with the subtree $r$.
  - $l\not= \texttt{Nil} \wedge l.\texttt{delMin}() = (l',k_{min}, v_{min}) \;\rightarrow
      \texttt{Node}(k, v, l, r).\texttt{delMin}() = \bigl(\texttt{Node}(k, v, l', r), k_{min}, v_{min}\bigr)$.

    If the left subtree $l$ in the binary tree $t = \texttt{Node}(k, v, l, r)$
    is not empty, then the smallest key of  $t$ is located inside the left subtree $l$.
    This smallest key is recursively removed from  $l$. This yields the tree 
    $l'$.  Next,  $l$ is replaced by $l'$ in $t$.  The resulting tree is
    $t' = \texttt{Node}(k, v, l', r)$.

In [ ]:
OrderedBinaryTree.prototype._delMin = function<Key, Value>(this: OrderedBinaryTree<Key, Value>): [OrderedBinaryTree<Key, Value>, Key, Value] {
    if (this.mLeft.isEmpty()) {
        return [this.mRight, this.mKey, this.mValue];
    } else {
        const [ls, km, vm] = this.mLeft!._delMin();
        this.mLeft = ls;
        return [this, km, vm];
    }
};

Given two ordered binary trees `s` and `t`, the expression `s._update(t)` overwrites the attributes of `s` with the corresponding attributes of `t`.

In [ ]:
OrderedBinaryTree.prototype._update = function<Key, Value>(this: OrderedBinaryTree<Key, Value>, t: OrderedBinaryTree<Key, Value>): void {
    this.mKey = t.mKey;
    this.mValue = t.mValue;
    this.mLeft = t.mLeft;
    this.mRight = t.mRight;
};

Given an ordered binary tree $b$, the method $b.\texttt{keyArray}()$ returns the array of all keys occurring in $b$.
Note that this array has to be sorted ascendingly.

In [ ]:
OrderedBinaryTree.prototype._keyArray = function<Key, Value>(this: OrderedBinaryTree<Key, Value>): Key[] {
    if (this.isEmpty()) {
        return [];
    }
    return [...this.mLeft._keyArray(), this.mKey, ...this.mRight._keyArray()];
};

In [ ]:
import { Graphviz } from "@hpcc-js/wasm";

Given a binary tree `t` the method `t._assignIDs(NodeDict)` assigns a unique identifier with each node.  The dictionary `NodeDict` maps these identifiers to the nodes where they occur.

In [ ]:
OrderedBinaryTree.prototype._assignIDs = function<Key, Value>(this: OrderedBinaryTree<Key, Value>, nodeDict: Record<number, OrderedBinaryTree<Key, Value>>): void {
    OrderedBinaryTree.sNodeCount += 1;
    this.mID = OrderedBinaryTree.sNodeCount;
    nodeDict[this.mID] = this;
    if (this.isEmpty()) {
        return;
    }
    this.mLeft._assignIDs(nodeDict);
    this.mRight._assignIDs(nodeDict);
};

Given an ordered binary tree $t$, the function $t.\texttt{toDot}()$ renders the tree graphically using `Graphviz`.

In [ ]:
OrderedBinaryTree.prototype.toDot = function<Key, Value>(this: OrderedBinaryTree<Key, Value>): string {
    OrderedBinaryTree.sNodeCount = 0;
    let dot = `digraph G {\nnode [shape=record style=rounded];\n`;
    const nodeDict: Record<number, OrderedBinaryTree<Key, Value>> = {};
    this._assignIDs(nodeDict);
    
    for (const [n, t] of Object.entries(nodeDict)) {
        if (t.mValue !== null) {
            dot += `${n} [label="{${t.mKey}|${t.mValue}}"];\n`;
        } else if (t.mKey !== null) {
            dot += `${n} [label="${t.mKey}"];\n`;
        } else {
            dot += `${n} [label="" shape=point];\n`;
        }
    }
    
    for (const [n, t] of Object.entries(nodeDict)) {
        if (t.mLeft !== null) {
            dot += `${n} -> ${t.mLeft.mID};\n`;
        }
        if (t.mRight !== null) {
            dot += `${n} -> ${t.mRight.mID};\n`;
        }
    }
    
    dot += `}\n`;
    return dot;
};

The function $\texttt{demo}()$ creates a small ordered binary tree.

In [ ]:
function demo(): OrderedBinaryTree<string, number> {
    const m = new OrderedBinaryTree<string, number>();
    m.insert("anton", 123);
    m.insert("hugo", 345);
    m.insert("gustav", 789);
    m.insert("mariam", 345);
    m.insert("jens", 234);
    m.insert("hubert", 432);
    m.insert("andre", 342);
    m.insert("philipp", 342);
    m.insert("rene", 345);
    m.insert("ans", 123);
    m.insert("alfa", 123);
    m.insert("algo", 345);
    return m;
}

In [ ]:
import { display } from "tslab";

async function renderTree<Key, Value>(tree: OrderedBinaryTree<Key, Value>): Promise<void> {
    const dot = await tree.toDot();
    const gv = await Graphviz.load();
    const svg = gv.layout(dot, "svg", "dot");
    display.html(svg);
}

In [ ]:
const t = demo();
await renderTree(t);

In [ ]:
t.insert('maurice', 123);
await renderTree(t);

In [ ]:
t.delete('mariam');
await renderTree(t);

In [ ]:
t.delete('anton');
await renderTree(t);

In [ ]:
t.delete('gustav');
await renderTree(t);

In [ ]:
t.delete('hubert');
await renderTree(t);

Let's generate an ordered binary tree with random keys.

In [ ]:
const t = new OrderedBinaryTree<number, null>();
const rnd = {
    range: (max: number) => Math.floor(Math.random() * max)
};
for (let i = 0; i < 30; i++) {
    const k = rnd.range(100);
    t.insert(k, null);
}
await renderTree(t);

This tree looks more or less balanced.  Lets us create a tree where things do not work out that well.

In [ ]:
const t = new OrderedBinaryTree<number, null>();
for (let k = 0; k < 30; k++) {
    t.insert(k, null);
}
await renderTree(t);

In order to check whether the method `delete` works as expected, we try the following:

In [ ]:
for (let k = 0; k < 30; k++) {
    t.delete(k);
}
await renderTree(t);

Let us compute the set $S$ of prime numbers up to some given number $n\in\mathbb{N}$.  Mathematically, this set can be defined as
$$ S = \{ 2, \cdots, n \} - \bigl\{ p \cdot q \;\big|\; p, q \in \{2, \cdots, n \} \bigr\}. $$

In [ ]:
const n = 100;
const S = new OrderedBinaryTree<number, null>();
const L = Array.from({ length: n - 1 }, (_, i) => i + 2);
function shuffle(array: number[]): void {
    for (let i = array.length - 1; i > 0; i--) {
        const j = Math.floor(Math.random() * (i + 1));
        [array[i], array[j]] = [array[j], array[i]];
    }
}
shuffle(L);
for (const x of L) {
    S.insert(x, null);
}
await renderTree(S);

In [ ]:
for (let p = 2; p <= Math.floor(n / 2); p++) {
    for (let q = p; q <= Math.floor(n / p); q++) {
        S.delete(p * q);
    }
}
await renderTree(S);

In [ ]:
console.log(S._keyArray());